Query Expansion Technique Implementation

In [12]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import TextLoader
from langchain.text_splitter import  RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [13]:
# Load and split data

loader = TextLoader('sample.txt')
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [14]:
#chunks

In [15]:
# Create vectorstore
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embedding_model)

#Create MMR Retriever
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001ADFEE6F490>, search_type='mmr', search_kwargs={'k': 5})

In [16]:
# Create LLM and Prompt
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

llm = init_chat_model('openai:o4-mini')

llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001AD9AEB98D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AD9AEB9ED0>, root_client=<openai.OpenAI object at 0x000001AD9AEB8B90>, root_async_client=<openai.AsyncOpenAI object at 0x000001AD9AEBB210>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [17]:
# Create Query Expansion

query_expansion_prompt = PromptTemplate.from_template(
    """
    You are a helpful assistant. Expand the following query to improve document retrieved by adding relevent lekely words and technical terms 
    
    Original query: '{query}'
    
    Expanded queries:
    1.
    2.
    3.
    4.
    5.
    """
)

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()

query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="\n    You are a helpful assistant. Expand the following query to improve document retrieved by adding relevent lekely words and technical terms \n\n    Original query: '{query}'\n\n    Expanded queries:\n    1.\n    2.\n    3.\n    4.\n    5.\n    ")
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001AD9AEB98D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AD9AEB9ED0>, root_client=<openai.OpenAI object at 0x000001AD9AEB8B90>, root_async_client=<openai.AsyncOpenAI object at 0x000001AD9AEBB210>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [18]:
query_expansion_chain.invoke({'query': 'Langchain memory'})

'Here are five expanded queries that add relevant technical terms and context around “LangChain memory”:\n\n1. LangChain memory management stateful context short-term long-term memory modules persistent memory vectorstore retrieval  \n2. LangChain conversational AI memory: ConversationBufferMemory, ConversationSummaryMemory, token buffer, memory chains  \n3. Implementing LangChain memory adapters (Redis, Chroma, Pinecone) with embedding-based memory for chatbots  \n4. LangChain memory architecture: memory variables, memory profiles, summarization memory, context window management  \n5. Best practices for LangChain memory in RAG pipelines and agents—dynamic memory, retrieval-augmented generation, memory strategies'

In [19]:
# RAG anwering prompt
answer_prompt = PromptTemplate.from_template(
    """
    Answer the question based on the context below.
    
    Context: {context}
    
    Question: {input}
    """
)

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=answer_prompt
)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the question based on the context below.\n\n    Context: {context}\n\n    Question: {input}\n    ')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001AD9AEB98D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AD9AEB9ED0>, root_client=<openai.OpenAI object at 0x000001AD9AEB8B90>, root_async_client=<openai.AsyncOpenAI object at 0x000001AD9AEBB210>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [20]:
# Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        'input': lambda x: x['input'],
        'context': lambda x: retriever.invoke(query_expansion_chain.invoke({'query': x['input']}))
    })
    | document_chain
)

In [21]:
# Run Query
query = {
    'input': 'What type of memory langchain support?'
}
print(query_expansion_chain.invoke({'query': query}))

response = rag_pipeline.invoke(query)
print(f'Answer:\n {response} ')

1. “What memory mechanisms and abstractions does LangChain support (e.g., buffer memory, summary memory, key-value memory, and entity memory) for conversational AI?”  
2. “Which persistent memory backends and vector stores are compatible with LangChain (such as Redis, Pinecone, FAISS, Chroma, and Weaviate) for storing and retrieving chat history?”  
3. “How does LangChain implement short-term vs. long-term memory, including sliding-window context buffers, conversation summarization modules, and embedding-based memory retrieval?”  
4. “What memory management patterns and modules are available in LangChain for Retrieval-Augmented Generation (RAG), such as conversational chain memory, dense vector indexing, and metadata filtering?”  
5. “Can you describe the built-in memory classes in LangChain (ConversationBufferMemory, ConversationSummaryMemory, CombinedMemory, etc.) and how they integrate with various LLM chains and external knowledge stores?”
Answer:
 LangChain comes with a built-in “

In [22]:
# Run Query
query = {
    'input': 'What type of memory CrewAI support?'
}
print(query_expansion_chain.invoke({'query': query}))

response = rag_pipeline.invoke(query)
print(f'Answer:\n {response} ')

Here are five ways you might expand that query with extra context, keywords and technical terms to retrieve more relevant documentation:

1. “What types of RAM modules does CrewAI support (DDR4, DDR5, ECC vs. non-ECC), and what are the maximum capacity and per-slot limits?”  
2. “CrewAI memory architecture details: supported memory channels (dual-channel, quad-channel), maximum bandwidth (GB/s), and recommended DIMM configurations.”  
3. “What are the onboard cache and main memory specifications for CrewAI? Include details on L1/L2/L3 cache sizes and supported memory frequencies.”  
4. “CrewAI system memory requirements and compatibility: supported form factors (UDIMM, SO-DIMM), memory speed ratings (MHz/MT/s), and vendor interoperability.”  
5. “Documentation on CrewAI memory subsystem: supported interfaces (DDR4-3200, DDR5-4800, LPDDR4x/LPDDR5), ECC support, error correction features, and expansion slots.”
Answer:
 CrewAI uses an “agentic memory” system—allowing the agent crew to rem

In [23]:
# Run Query
query = {
    'input': 'How can you relate python to langchain, crewai and agentic ai?'
}
print(query_expansion_chain.invoke({'query': query}))

response = rag_pipeline.invoke(query)
print(f'Answer:\n {response} ')

Here are five expanded query formulations that add relevant keywords and technical terms to surface richer documentation:

1. “How do you use Python to build and orchestrate language model pipelines with LangChain, integrate CrewAI multi-agent workflows, and implement agentic AI architectures (e.g., autonomous LLM agents, reinforcement-learning loops)?”

2. “What are best practices for Python-based development with LangChain’s prompt templates, chain-of-thought prompting, embedding retrieval (RAG), combined with CrewAI task orchestration and agentic AI self-management?”

3. “Can you demonstrate Python code examples showing end-to-end integration of LangChain (chains, LLM wrappers, vectorstores), CrewAI multi-agent coordination, and agentic AI features such as tool use and memory?”

4. “How does Python interact with LangChain’s modular components (prompts, agents, memory) alongside CrewAI’s workflow engine to realize agentic AI systems with dynamic planning, decision making, and API too